<a href="https://colab.research.google.com/github/laurencushing/steam-smart-shopper/blob/main/steam_smart_shopper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Smart Shopper

My app helps gamers on a budget decide which Steam games to buy by comparing prices, discounts, player ratings, and average playtime.



In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/


In [ ]:
print(games.columns.tolist())

['appid', 'name', 'release_date', 'release_year', 'price', 'discount', 'required_age', 'windows', 'mac', 'linux', 'developers', 'publishers', 'genres', 'n_languages', 'achievements', 'metacritic_score', 'positive', 'negative', 'pct_pos_total', 'num_reviews_total', 'estimated_owners', 'owners_low', 'avg_playtime_hours', 'peak_ccu', 'short_description', 'critic_reviews']


In [ ]:
print(games.columns.tolist()[10:])

['developers', 'publishers', 'genres', 'n_languages', 'achievements', 'metacritic_score', 'positive', 'negative', 'pct_pos_total', 'num_reviews_total', 'estimated_owners', 'owners_low', 'avg_playtime_hours', 'peak_ccu', 'short_description', 'critic_reviews']


In [ ]:
def find_games(games, budget):
    matches = games[
        (games["price"] <= budget) &
        (games["metacritic_score"] >= 80)
    ]
    return matches[["name", "price", "metacritic_score"]].head(5)

print(find_games(games, 20))

                        name  price  metacritic_score
2                     Dota 2   0.00                90
3  Grand Theft Auto V Legacy   0.00                96
5            Team Fortress 2   0.00                92
6                   Terraria   9.99                83
9              Apex Legends™   0.00                88


In [4]:
def find_deals(games, budget):
  deals = games [
      (games["price"] <= budget) &
      (games["discount"] > 0)
  ]
  return deals[["name", "price", "discount"]].head(5)

print(find_deals(games, 20))

                                name  price  discount
4    Tom Clancy's Rainbow Six® Siege   3.99        80
8                        Garry's Mod   5.99        40
21                          Among Us   2.99        40
104                       Far Cry® 5   5.99        90
105        Assassin's Creed® Odyssey  11.99        80


In [7]:
print(games[["name", "peak_ccu"]].head(10))

                              name  peak_ccu
0                 Counter-Strike 2   1212356
1              PUBG: BATTLEGROUNDS    616738
2                           Dota 2    555977
3        Grand Theft Auto V Legacy    117698
4  Tom Clancy's Rainbow Six® Siege     89916
5                  Team Fortress 2     50817
6                         Terraria     30516
7                             Rust    200902
8                      Garry's Mod     32384
9                    Apex Legends™    151844


In [5]:
print(games.columns.tolist())

['appid', 'name', 'release_date', 'release_year', 'price', 'discount', 'required_age', 'windows', 'mac', 'linux', 'developers', 'publishers', 'genres', 'n_languages', 'achievements', 'metacritic_score', 'positive', 'negative', 'pct_pos_total', 'num_reviews_total', 'estimated_owners', 'owners_low', 'avg_playtime_hours', 'peak_ccu', 'short_description', 'critic_reviews']


In [6]:
def find_popular(games, budget):
  popular = games [
      (games["price"] <= budget) &
      (games["metacritic_score"] >= 80)
  ]
  popular = popular.sort_values("peak_ccu", ascending=False)
  return popular[["name", "price", "metacritic_score", "peak_ccu"]].head(5)

print(find_popular(games, 20))

                         name  price  metacritic_score  peak_ccu
2                      Dota 2   0.00                90    555977
9               Apex Legends™   0.00                88    151844
3   Grand Theft Auto V Legacy   0.00                96    117698
13             Stardew Valley  14.99                89     67309
23                War Thunder   0.00                81     52069


In [9]:
import requests

url = "https://www.cheapshark.com/api/1.0/deals"

response = requests.get(url, headers={"User-Agent": "SteamSmartShopper/1.0"})

print(response)

<Response [200]>


In [10]:
deals = response.json()

print(deals[0]["title"])
print(deals[0]["salePrice"])
print(deals[0]["normalPrice"])
print(deals[0]["savings"])



Ashes of the Singularity II
25.91
39.99
35.208802


In [11]:
game_id = deals[0]["gameID"]

url = "https://www.cheapshark.com/api/1.0/games"

response = requests.get(
    url,
    params={"id": game_id},
    headers={"User-Agent": "SteamSmartShopper/1.0"}
)

game_info = response.json()

print(game_info["cheapestPriceEver"])

{'price': '25.89', 'date': 1794419249}


In [13]:
sale_price = float(deals[0]["salePrice"])
lowest_price = float(game_info["cheapestPriceEver"]["price"])

difference = sale_price - lowest_price

print("Game:", deals[0]["title"])
print("Sale price: $, sale_price")
print("Historical low: $", lowest_price)
print("Difference: $", round(difference, 2))

Game: Ashes of the Singularity II
Sale price: $, sale_price
Historical low: $ 25.89
Difference: $ 0.02


In [14]:
def compare_prices(deal, game_info):
  sale_price = float(deal["salePrice"])
  lowest_price = float(game_info["cheapestPriceEver"]["price"])

  difference = round(sale_price - lowest_price, 2)

  print("Game:", deal["title"])
  print("Sale price: $", sale_price)
  print("Historical low: $", lowest_price)
  print("Historical: $", difference)

compare_prices(deals[0], game_info)

Game: Ashes of the Singularity II
Sale price: $ 25.91
Historical low: $ 25.89
Historical: $ 0.02


In [15]:
for deal in deals[:3]:
  game_id = deal["gameID"]

  response = requests.get(
      "https://www.cheapshark.com/api/1.0/games",
      params={"id": game_id},
      headers={"User-Agent": "SteamSmartShopper/1.0"}
  )

  game_info = response.json()

  compare_prices(deal, game_info)
  print()

Game: Ashes of the Singularity II
Sale price: $ 25.91
Historical low: $ 25.89
Historical: $ 0.02

Game: Need for Speed Heat Deluxe Edition
Sale price: $ 3.49
Historical low: $ 3.49
Historical: $ 0.0

Game: Battlefield 2042 Elite Edition
Sale price: $ 4.49
Historical low: $ 4.49
Historical: $ 0.0

